# BEE 4750 Homework 5: Gaussian Plumes and Model Validation

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 10/22/26, 9:00pm

## Overview

### Instructions

- Problem 1 involves evaluating the plume equation and reasoning about
  stability class.
- Problem 2 asks you to analyze residuals to interpret goodness-of-fit.
- Problem 3 gives you a working plume model and a set of monitor
  observations, and asks you to interpret the results.

This assignment is deliberately short on implementation. The work is in
the interpretation.

### Load Environment

In [0]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

In [0]:
using CSV
using DataFrames
using Statistics
using Optim
using Plots

## Problems (Total: 50 Points)

### Problem 1 (10)

The ground-level concentration downwind of an elevated point source, on
the plume centreline and accounting for reflection at the ground, is

$$C(x, y) = \frac{Q}{\pi u \sigma_y(x) \sigma_z(x)} \exp\left(\frac{-y^2}{2\sigma_y(x)^2}\right)\exp\left(\frac{-H^2}{2\sigma_z(x)^2}\right).$$

A stack emits $Q = 100$ g/s at an effective height of $H = 41$ m into a
wind of $u = 2.5$ m/s.

#### Problem 1.1 (5)

At 2 km downwind under the stability class in question, $\sigma_y = 62$
m and $\sigma_z = 23$ m.

1.  Compute the ground-level concentration on the centreline at 2 km, in
    µg/m³.
2.  Compute it again at a receptor 60 m off the centreline at the same
    distance. What fraction of the centreline value is it?

#### Problem 1.2 (5)

Stability classes run from A (very unstable, strong daytime heating)
through F (stable, clear calm night). In lecture, this same stack
produced a higher ground-level peak under class B (a sunny afternoon)
than under class F (a clear night), much closer to the stack. The
dispersion coefficients at two distances downwind are:

| Distance | Class B: $\sigma_y$, $\sigma_z$ (m) | Class F: $\sigma_y$, $\sigma_z$ (m) |
|:-----------------------|-----------------------:|-----------------------:|
| 1 km | 156, 110 | 34, 14 |
| 8 km | 1001, 1064 | 218, 42 |

A school sits 1 km directly downwind of the stack, and a town 8 km
downwind. The 1-hour SO$_2$ standard is 196 µg/m³.

1.  Compute the ground-level centreline concentration at the school and
    at the town under each class. Which class is the worst case at each
    receptor?
2.  The facility’s consultant proposes a permit analysis that models
    only class B, since it produces the highest peak. What would that
    analysis miss? Which conditions should a permit analysis model
    instead, and why?

### Problem 2 (5)

A dissolved-oxygen model is compared with measurements at six stations
along a river, numbered in order downstream. Three versions of the model
leave these residuals (model minus observation, in mg/L):

| Station |    1 |    2 |    3 |    4 |    5 |    6 |
|:--------|-----:|-----:|-----:|-----:|-----:|-----:|
| Model A | +0.7 | −0.5 | −0.1 | +0.5 | −0.7 | +0.1 |
| Model B | +0.5 | +0.5 | +0.5 | +0.5 | +0.5 | +0.5 |
| Model C | −0.7 | −0.5 | −0.1 | +0.1 | +0.5 | +0.7 |

#### Problem 2.1 (1)

Compute the RMSE of each model.

#### Problem 2.2 (3)

For each model, say in a sentence what its residuals show that the RMSE
does not, and what kind of error that suggests.

#### Problem 2.3 (1)

A parameter that shifts every prediction up or down by the same amount
(for example, a sensor’s calibration offset) is changed until the fit is
as good as it can be. Which model or models could that fix?

### Problem 3 (35)

An industrial facility has been monitored for a night’s operation. You
are given twelve monitor readings in `data/receptors.csv`, and a working
implementation of the plume model below.

The operator reports that conditions that night were **stability class
F**, and the model below is configured accordingly. Your job is to
decide whether that model is fit to be used.

In [0]:
# Pasquill-Gifford coefficients for class F. Distances in km, sigmas in metres.
class_F = (; a=34.0, c_near=14.35, d_near=0.740, f_near=-0.35,
             c_far=62.6, d_far=0.180, f_far=-48.6)

sigma_y(downwind_km, coef) = coef.a * downwind_km^0.894

function sigma_z(downwind_km, coef)
    if downwind_km < 1.0
        return coef.c_near * downwind_km^coef.d_near + coef.f_near
    else
        return coef.c_far * downwind_km^coef.d_far + coef.f_far
    end
end

function ground_concentration(downwind_km, crosswind_m, coef;
                              emission=100.0, height=41.0, wind=2.5)
    spread_y = sigma_y(downwind_km, coef)
    spread_z = sigma_z(downwind_km, coef)
    if downwind_km <= 0.05 || spread_z <= 0.0
        return 0.0
    end
    return (emission / (π * wind * spread_y * spread_z)) *
        exp(-crosswind_m^2 / (2 * spread_y^2)) *
        exp(-height^2 / (2 * spread_z^2)) * 1e6
end

monitors = CSV.read("data/receptors.csv", DataFrame)

#### Problem 3.1 (8)

Run the model at each monitor location and compute the RMSE against the
observations. Report it both in µg/m³ and as a percentage of the mean
observed concentration.

On that number alone, would you say the model fits?

#### Problem 3.2 (9)

Plot the residuals (model minus observation) against downwind distance.

1.  Describe the pattern. Is it scatter, or is there a pattern?
2.  What does the pattern tell you that the RMSE does not?
3.  Based on the pattern alone, is the model’s error more consistent
    with *noisy observations*, with a *wrong parameter value*, or with
    the *wrong model structure*? Justify your answer.

#### Problem 3.3 (9)

Wind speed is the most uncertain input here, and the whole concentration
field scales as $1/u$. It is the obvious thing to calibrate.

Fit the wind speed by minimising RMSE against the observations, holding
everything else fixed.

1.  Report the fitted wind speed and the RMSE after calibration.
2.  Did calibration meaningfully improve the fit?
3.  Explain the result in terms of what wind speed *can* and *cannot*
    change about the modelled field. Your answer should refer to the
    residual pattern from Problem 3.2.

In [0]:
function rmse_for_wind(wind_speed)
    # TODO: run the model at every monitor with this wind speed and return
    # the RMSE against the observed concentrations.
    error("rmse_for_wind has not been written yet -- replace this line")
end

#### Problem 3.4 (9)

You are asked to certify this model as fit for use in siting a permanent
compliance monitor at the facility.

1.  Would you certify it? State your decision plainly.
2.  The fit statistics are what they are, and you have reported them.
    What else do you owe the person who will rely on this model, beyond
    the numbers you computed?
3.  Name one specific check, measurement, or piece of information you
    would want before certifying, and say what it would settle.
4.  Suppose the operator pushes back and points out that no parameter
    you fitted made the model better, so the model is “as good as it can
    be.” Respond to that argument.

## References

List any external references consulted, including classmates.